In [1]:
!uv pip install wandb python-dotenv

Using Python 3.12.9 environment at: /panfs/g52-panfs/exp/venv/da32459/.conda/envs/jnb
Audited 2 packages in 6ms


In [2]:
import os
from dotenv import load_dotenv

load_dotenv()

wandb_project_name = 'coco-yolov8'
os.environ['WANDB_PROJECT'] = wandb_project_name
os.environ['WANDB_MODE'] = 'online'

In [3]:
import wandb

wandb.login()

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: dygwon (dygwon-personal) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [4]:
from ultralytics import settings
print(settings)

JSONDict("/home/da32459/.config/Ultralytics/settings.json"):
{
  "settings_version": "0.0.6",
  "datasets_dir": "/panfs/g52-panfs/scratch/da32459/repos/cs556/datasets",
  "weights_dir": "weights",
  "runs_dir": "runs",
  "uuid": "683a6faf8dc0b48240b69dce2692c4434f09c63a32ed92a4d542baa419551c3f",
  "sync": true,
  "api_key": "",
  "openai_api_key": "",
  "clearml": true,
  "comet": true,
  "dvc": true,
  "hub": true,
  "mlflow": true,
  "neptune": true,
  "raytune": true,
  "tensorboard": false,
  "wandb": false,
  "vscode_msg": true,
  "openvino_msg": true
}


In [5]:
settings.update({'wandb': True})

In [6]:
import torch

print(f'CUDA available: {torch.cuda.is_available()}')
print(f'Device: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

CUDA available: True
Device: NVIDIA H200
VRAM: 150.1 GB


In [7]:
import fiftyone as fo
import fiftyone.zoo as foz

In [8]:
CLASSES = ["car", "motorcycle", "bus", "truck"]

In [9]:
# Load the COCO-2017 splits you need.
# For a quick test, add max_samples=1000 to limit the download.
train_dataset = foz.load_zoo_dataset(
    "coco-2017",
    split="train",
    dataset_name="coco-2017-train",
    # max_samples=1000,  # uncomment to limit for faster iteration
    classes=CLASSES,
)

val_dataset = foz.load_zoo_dataset(
    "coco-2017",
    split="validation",
    dataset_name="coco-2017-val",
    # max_samples=500,
    classes=CLASSES,
)

print(f"Train samples: {len(train_dataset)}")
print(f"Val samples:   {len(val_dataset)}")

Found annotations at '/home/da32459/fiftyone/coco-2017/raw/instances_train2017.json'
Sufficient images already downloaded
Existing download of split 'train' is sufficient
Loading 'coco-2017' split 'train'
 100% |█████████████| 18175/18175 [1.4m elapsed, 0s remaining, 243.2 samples/s]      
Dataset 'coco-2017-train' created
Found annotations at '/home/da32459/fiftyone/coco-2017/raw/instances_val2017.json'
Sufficient images already downloaded
Existing download of split 'validation' is sufficient
Loading 'coco-2017' split 'validation'
 100% |█████████████████| 796/796 [3.4s elapsed, 0s remaining, 243.0 samples/s]      
Dataset 'coco-2017-val' created
Train samples: 18175
Val samples:   796


In [10]:
import os

EXPORT_DIR = "/scratch/da32459/repos/cs556/coco_yolo"

# FiftyOne can export directly to YOLOv5-format (works with YOLOv8)
train_dataset.export(
    export_dir=EXPORT_DIR,
    dataset_type=fo.types.YOLOv5Dataset,
    split="train",
    label_field="ground_truth",   # COCO zoo dataset uses this field
    classes=CLASSES
)

val_dataset.export(
    export_dir=EXPORT_DIR,
    dataset_type=fo.types.YOLOv5Dataset,
    split="val",
    label_field="ground_truth",
    classes=CLASSES,
)

print("Exported to:", EXPORT_DIR)
print(os.listdir(EXPORT_DIR))

Directory '/scratch/da32459/repos/cs556/coco_yolo' already exists; export will be merged with existing files
   0% |-------------|     3/18175 [326.8ms elapsed, 33.0m remaining, 9.2 samples/s] 

/panfs/g52-panfs/exp/venv/da32459/.conda/envs/jnb/lib/python3.12/site-packages/fiftyone/utils/yolo.py:1154: UserWarning: Ignoring object with label 'bicycle' not in provided classes
  warnings.warn(msg)
/panfs/g52-panfs/exp/venv/da32459/.conda/envs/jnb/lib/python3.12/site-packages/fiftyone/utils/yolo.py:1154: UserWarning: Ignoring object with label 'person' not in provided classes
  warnings.warn(msg)
/panfs/g52-panfs/exp/venv/da32459/.conda/envs/jnb/lib/python3.12/site-packages/fiftyone/utils/yolo.py:1154: UserWarning: Ignoring object with label 'traffic light' not in provided classes
  warnings.warn(msg)


   0% ||------------|     9/18175 [538.9ms elapsed, 18.1m remaining, 16.7 samples/s] 

/panfs/g52-panfs/exp/venv/da32459/.conda/envs/jnb/lib/python3.12/site-packages/fiftyone/utils/yolo.py:1154: UserWarning: Ignoring object with label 'wine glass' not in provided classes
  warnings.warn(msg)
/panfs/g52-panfs/exp/venv/da32459/.conda/envs/jnb/lib/python3.12/site-packages/fiftyone/utils/yolo.py:1154: UserWarning: Ignoring object with label 'bench' not in provided classes
  warnings.warn(msg)
/panfs/g52-panfs/exp/venv/da32459/.conda/envs/jnb/lib/python3.12/site-packages/fiftyone/utils/yolo.py:1154: UserWarning: Ignoring object with label 'skateboard' not in provided classes
  warnings.warn(msg)


   0% |-------------|    17/18175 [763.8ms elapsed, 13.6m remaining, 22.3 samples/s] 

...

/panfs/g52-panfs/exp/venv/da32459/.conda/envs/jnb/lib/python3.12/site-packages/fiftyone/utils/yolo.py:1154: UserWarning: Ignoring object with label 'oven' not in provided classes
  warnings.warn(msg)


  10% |█|-----------|  1834/18175 [43.7s elapsed, 6.4m remaining, 46.8 samples/s]    

/panfs/g52-panfs/exp/venv/da32459/.conda/envs/jnb/lib/python3.12/site-packages/fiftyone/utils/yolo.py:1154: UserWarning: Ignoring object with label 'sink' not in provided classes
  warnings.warn(msg)


  17% |██\----------|  3000/18175 [1.2m elapsed, 5.8m remaining, 45.1 samples/s]     


KeyboardInterrupt: 

In [12]:
import yaml

dataset_yaml = {
    "path": EXPORT_DIR,
    "train": "images/train",
    "val": "images/val",
    "nc": len(CLASSES),
    "names": CLASSES,
}

yaml_path = os.path.join(EXPORT_DIR, "dataset.yaml")
with open(yaml_path, "w") as f:
    yaml.dump(dataset_yaml, f, default_flow_style=False)

print(f"Dataset YAML written to {yaml_path}")
print(f"Number of classes: {len(CLASSES)}")

Dataset YAML written to /scratch/da32459/repos/cs556/coco_yolo/dataset.yaml
Number of classes: 4


In [13]:
from ultralytics import YOLO

# Pick a model size: yolov8n, yolov8s, yolov8m, yolov8l, yolov8x
model = YOLO("yolov8m.pt")
print('model loaded')

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/home/da32459/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
model loaded


In [12]:
run_name = 'run-yolov8m-full'

In [ ]:
results = model.train(
    data=yaml_path,
    epochs=100,           # increase for better results
    imgsz=640,
    batch=96,            # let ultralytics decide optimal
    device=0,            # 0 = first GPU
    project=wandb_project_name,
    name=run_name,
    save=True,
    save_period=10,          # checkpoint every 10 epochs
    patience=10,         # early stopping
    workers=16,           # Colab has limited CPU cores

    optimizer="AdamW",
    lr0=3.5e-3,
    lrf=0.05,                # final lr = lr0 * lrf
    cos_lr=True,
    warmup_epochs=3,
    weight_decay=5e-4,
    amp=True,                # automatic mixed precision (BF16 on A100) — free speedup

    close_mosaic=10,
    mosaic=1.0,
    mixup=0.1,
    copy_paste=0.1,          # good for occluded vehicle scenarios

    # Augmentation (helps generalize to varied vehicle appearances)
    fliplr=0.5,
    flipud=0.0,
    hsv_h=0.02,
    hsv_s=0.7,
    hsv_v=0.4,
)

Ultralytics 8.4.19 🚀 Python-3.12.9 torch-2.10.0+cu128 CUDA:0 (NVIDIA H200, 143158MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=96, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.1, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/scratch/da32459/repos/cs556/coco_yolo/dataset.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.02, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0035, lrf=0.05, mask_ratio=4, max_det=300, mixup=0.1, mode=train, model=yolov8m.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=run-yolov8m-full, nbs=64, nms=False, opset=None, optimize=False, optimizer=AdamW, overlap_m

/panfs/g52-panfs/exp/venv/da32459/.conda/envs/jnb/lib/python3.12/site-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 32 worker processes in total. Our suggested max number of worker in current system is 16, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/panfs/g52-panfs/exp/venv/da32459/.conda/envs/jnb/lib/python3.12/site-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 32 worker processes in total. Our suggested max number of worker in current system is 16, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
 

optimizer: AdamW(lr=0.0035, momentum=0.937) with parameter groups 77 weight(decay=0.0), 84 weight(decay=0.00075), 83 bias(decay=0.0)
Plotting labels to /panfs/g52-panfs/scratch/da32459/repos/cs556/runs/detect/coco-yolov8/run-yolov8m-full/labels.jpg... 
Image sizes 640 train, 640 val
Using 16 dataloader workers
Logging results to /panfs/g52-panfs/scratch/da32459/repos/cs556/runs/detect/coco-yolov8/run-yolov8m-full
Starting training for 100 epochs...

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      1/100      34.9G      1.339      1.745      1.347        227        640: 100% ━━━━━━━━━━━━ 190/190 2.9it/s 1:060.3sss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 5/5 1.1s/it 5.4s0.6s0s
                   all        796       3003      0.131      0.212     0.0629     0.0312

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      2/100      43.5G      1.499      1.806 

In [ ]:
metrics = model.val(data=yaml_path)
print(f"mAP50:    {metrics.box.map50:.4f}")
print(f"mAP50-95: {metrics.box.map:.4f}")

In [ ]:
run = wandb.init(
    project=wandb_project_name,
    job_type='upload'
)

artifact = wandb.Artifact(run_name, type='model')
artifact.add_file(f'/scratch/da32459/repos/cs556/runs/detect/{wandb_project_name}/{run_name}/weights/best.pt')

In [14]:
from ultralytics import YOLO

EXPORT_DIR = "/scratch/da32459/repos/cs556/coco_yolo"
yaml_path = os.path.join(EXPORT_DIR, "dataset.yaml")

val_dataset = fo.load_dataset("coco-2017-val")
best_ckpt = f'/scratch/da32459/repos/cs556/runs/detect/{wandb_project_name}/{run_name}/weights/best.pt'
trained_model = YOLO(best_ckpt)

metrics = trained_model.val(data=yaml_path)
print(f"mAP50:    {metrics.box.map50:.4f}")
print(f"mAP50-95: {metrics.box.map:.4f}")

Ultralytics 8.4.19 🚀 Python-3.12.9 torch-2.10.0+cu128 CUDA:0 (NVIDIA H200, 143158MiB)
Model summary (fused): 93 layers, 25,842,076 parameters, 0 gradients, 78.7 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2650.7±629.0 MB/s, size: 154.8 KB)
val: Scanning /panfs/g52-panfs/scratch/da32459/repos/cs556/coco_yolo/labels/val.cache... 796 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 796/796 208.7Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 50/50 15.6it/s 3.2s0.1s
                   all        796       3003      0.763      0.668      0.742      0.546
                   car        535       1932      0.755       0.69      0.746      0.513
            motorcycle        159        371      0.797      0.654      0.746      0.495
                   bus        189        285      0.867      0.779      0.858      0.721
                 truck        250        415      0.636      0.549      0.617      0.457
S

In [18]:
import fiftyone as fo

# Run inference on the val set
val_dataset = fo.load_dataset("coco-2017-val")
best_ckpt = f"/scratch/da32459/repos/cs556/runs/detect/{wandb_project_name}/{run_name}/weights/best.pt"
trained_model = YOLO(best_ckpt)

for sample in val_dataset.iter_samples(autosave=True):
    result = trained_model(sample.filepath, verbose=False)[0]
    detections = []
    h, w = result.orig_shape
    for box, conf, cls in zip(
        result.boxes.xyxy.cpu().numpy(),
        result.boxes.conf.cpu().numpy(),
        result.boxes.cls.cpu().numpy(),
    ):
        x1, y1, x2, y2 = box
        detections.append(
            fo.Detection(
                label=result.names[int(cls)],
                bounding_box=[x1 / w, y1 / h, (x2 - x1) / w, (y2 - y1) / h],
                confidence=float(conf),
            )
        )
    sample["yolov8_preds"] = fo.Detections(detections=detections)

# Launch the FiftyOne app (works inline in Colab)
session = fo.launch_app(val_dataset, auto=False)

Session launched. Run `session.show()` to open the App in a cell output.


In [20]:
from fiftyone.utils.annotations import draw_labeled_images

draw_labeled_images(
    val_dataset.take(50),
    "/scratch/da32459/repos/cs556/viz_output",
    label_fields=["ground_truth", "yolov8_preds"],
)

 100% |███████████████████| 50/50 [4.4s elapsed, 0s remaining, 12.2 samples/s]      


['/scratch/da32459/repos/cs556/viz_output/000000523782.jpg',
 '/scratch/da32459/repos/cs556/viz_output/000000270066.jpg',
 '/scratch/da32459/repos/cs556/viz_output/000000478474.jpg',
 '/scratch/da32459/repos/cs556/viz_output/000000158548.jpg',
 '/scratch/da32459/repos/cs556/viz_output/000000001532.jpg',
 '/scratch/da32459/repos/cs556/viz_output/000000210394.jpg',
 '/scratch/da32459/repos/cs556/viz_output/000000463730.jpg',
 '/scratch/da32459/repos/cs556/viz_output/000000350023.jpg',
 '/scratch/da32459/repos/cs556/viz_output/000000276707.jpg',
 '/scratch/da32459/repos/cs556/viz_output/000000338905.jpg',
 '/scratch/da32459/repos/cs556/viz_output/000000137727.jpg',
 '/scratch/da32459/repos/cs556/viz_output/000000342128.jpg',
 '/scratch/da32459/repos/cs556/viz_output/000000144706.jpg',
 '/scratch/da32459/repos/cs556/viz_output/000000398742.jpg',
 '/scratch/da32459/repos/cs556/viz_output/000000347544.jpg',
 '/scratch/da32459/repos/cs556/viz_output/000000100283.jpg',
 '/scratch/da32459/repos